<a href="https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


30000 pages |  declining rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth a refresh review if it is **stale** (hasn't been updated in a long while) **and** it's still **visible** (real people are seeing it in search). Between two stale-and-visible pages, the one with more exposure is the bigger opportunity, so I rank by exposure inside the flagged group. This is the same `stale × visible` idea from `02_your_first_readable_model.ipynb`, but this time it carries one reason code and an action label instead of just a number.

Before coding it, I check the two signals it leans on — **staleness** and **visibility** — against the actual decline rate. One of these (staleness) sits directly behind FlyRank's real refresh flags, so it's the flag-linked check this card asks for.

In [2]:
# --- Signal check 1: staleness (flag-linked — this is the signal behind FlyRank's refresh flags) ---
# Claim: "the longer since a page was last updated, the more likely it is declining."
# freshness_tier is the repo's own bucketing of days_since_last_update (see docs/data-dictionary.md).
order1 = ["0-30", "31-90", "91-180", "181+"]
df["freshness_tier"] = pd.Categorical(df["freshness_tier"], categories=order1, ordered=True)

staleness_table = df.groupby("freshness_tier", observed=True)["is_declining_label"].agg(
    decline_rate="mean", n="count"
).round(3)
print("Signal 1 — staleness (freshness_tier) vs. decline rate")
print(staleness_table)
print(f"\nOverall base rate: {df['is_declining_label'].mean():.3f}")

Signal 1 — staleness (freshness_tier) vs. decline rate
                decline_rate      n
freshness_tier                     
0-30                   0.511  20480
31-90                  0.589    175
91-180                 0.611   9171
181+                   0.471    174

Overall base rate: 0.542


### **VERDICT:**
MIXED.
The two tiers that hold almost all the data DO move the right way: pages updated 0-30 days ago
decline 51.1% of the time (n=20,480, just under the 54.2% base rate) vs. 61.1% for pages stale
91-180 days (n=9,171, above base rate) -- staler content really is more likely to be declining.
But the two thin extreme tiers (31-90 days, n=175; 181+ days, n=174) do NOT continue that trend --
they sit close to or below the fresh tier's rate. Both thin tiers clear the ~50-row floor, so I
can't just wave them away as "too small to count", but with only 174-175 rows each against tiers
20-50x their size, a few dozen mislabeled pages would flip them. Net: staleness is a real, directional
signal where the data is thick, and I should not lean on it in the thin tail without more rows.

In [3]:
# --- Signal check 2: visibility (the other half of the rule) ---
# Claim: "pages that are more visible (more search impressions) are more likely declining."
# impression_tier is the repo's own bucketing of impressions_90d.
order2 = ["low", "moderate", "good", "excellent"]
df["impression_tier"] = pd.Categorical(df["impression_tier"], categories=order2, ordered=True)

visibility_table = df.groupby("impression_tier", observed=True)["is_declining_label"].agg(
    decline_rate="mean", n="count"
).round(3)
print("Signal 2 — visibility (impression_tier) vs. decline rate")
print(visibility_table)


Signal 2 — visibility (impression_tier) vs. decline rate
                 decline_rate      n
impression_tier                     
low                     0.454  11248
moderate                0.615  10469
good                    0.586   7205
excellent               0.462   1078



### **VERDICT:**
 MIXED.
This is NOT a clean line: low (45.4%, n=11,248) and excellent (46.2%, n=1,078) traffic pages both
decline LESS than moderate (61.5%, n=10,469) and good (58.6%, n=7,205) traffic pages -- a U-shape,
not a trend. So visibility on its own does not predict decline the way staleness does.
This is actually fine for my rule: I am not using "visible" as a decline predictor, I am using it
as a MATERIALITY gate -- "don't bother flagging a stale page nobody sees." Reading this table
straight, a clearly-explained negative here is exactly the point: it stopped me from overclaiming
what impressions_90d buys me, and confirmed it belongs in the rule as a filter, not as the reason
code's main driver.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Score: stale x visible x exposure (readable on purpose, no fitted weights).
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions_90d"]

# ONE reason code + one action label.
df["reason_code"] = np.where((stale == 1) & (visible == 1), "stale_but_visible", "not_flagged")
df["action"] = np.where(df["score"] > 0, "refresh_review", "no_action")

print("Flagged (score > 0):", (df['score'] > 0).sum(), "of", len(df), "pages")
print(df["reason_code"].value_counts())
print(df["action"].value_counts())


Flagged (score > 0): 17 of 30000 pages
reason_code
not_flagged          29983
stale_but_visible       17
Name: count, dtype: int64
action
no_action         29983
refresh_review       17
Name: count, dtype: int64


In [5]:
import os

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

out_cols = ["content_id", "client_id", "score", "reason_code", "action",
            "days_since_last_update", "impressions_90d", "avg_position", "ctr",
            "content_type", "trend_direction"]
# trend_direction rides along ONLY as a read-only audit column for the human review below --
# it is never an input to score, reason_code, or action.

os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv --", len(ranked), "rows")

ranked[out_cols].head(10)


Wrote work/outputs/baseline_action_score.csv -- 30000 rows


,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,content_type,trend_direction
0,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,refresh_review,194,61678,19.7,0.15,keyword article,down
1,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,refresh_review,194,59472,24.8,0.13,keyword article,down
2,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,refresh_review,194,25715,22.2,0.23,keyword article,down
3,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,refresh_review,193,13299,10.5,0.49,keyword article,down
4,content_5feee3994adb,client_7f2253d7e2,7812,stale_but_visible,refresh_review,194,7812,39.0,0.01,keyword article,down
5,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,refresh_review,193,7558,17.9,0.20,keyword article,down
6,content_b16bd7307b39,client_7f2253d7e2,4590,stale_but_visible,refresh_review,194,4590,31.0,0.00,keyword article,down
7,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,refresh_review,194,4556,16.4,0.33,keyword article,down
8,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,refresh_review,194,4429,25.3,0.38,keyword article,down
9,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,refresh_review,193,1697,15.8,0.12,keyword article,down


In [6]:
# Honest gut-check, same metric the session used (not required by the card, but free to compute).
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values
base_rate = y.mean()
for k in (10, 20, 50):
    print(f"Precision@{k}: {precision_at_k(df['score'], y, k):.3f}   (base rate: {base_rate:.3f})")


Precision@10: 1.000   (base rate: 0.542)
Precision@20: 0.900   (base rate: 0.542)
Precision@50: 0.680   (base rate: 0.542)


## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

All 17 pages the rule flags come from just 4 clients, and the raw-impressions ranking pushes one client's biggest pages (`client_7f2253d7e2`) to the very top of the top 10 — that's already visible below and I come back to it in Section 4.

In [7]:
top10 = ranked.head(10).reset_index(drop=True)

reviews = []
for i, row in top10.iterrows():
    reviews.append({
        "rank": i + 1,
        "content_id": row["content_id"],
        "action": row["action"],
        "why": (f"reason_code={row['reason_code']}: not updated in "
                f"{int(row['days_since_last_update'])} days, still pulling "
                f"{int(row['impressions_90d']):,} impressions/90d "
                f"(ctr={row['ctr']:.2f}%, avg_position={row['avg_position']:.1f})."),
        "what_would_make_it_wrong": None,  # filled by hand below, one line each
    })

for r in reviews:
    print(f"{r['rank']:>2}. {r['content_id']}  ({r['action']})")
    print(f"    why: {r['why']}")
    print()


 1. content_cf56e2e2e282  (refresh_review)
    why: reason_code=stale_but_visible: not updated in 194 days, still pulling 61,678 impressions/90d (ctr=0.15%, avg_position=19.7).

 2. content_7368877ea310  (refresh_review)
    why: reason_code=stale_but_visible: not updated in 194 days, still pulling 59,472 impressions/90d (ctr=0.13%, avg_position=24.8).

 3. content_1bfaa38ff26c  (refresh_review)
    why: reason_code=stale_but_visible: not updated in 194 days, still pulling 25,715 impressions/90d (ctr=0.23%, avg_position=22.2).

 4. content_0a91db491d14  (refresh_review)
    why: reason_code=stale_but_visible: not updated in 193 days, still pulling 13,299 impressions/90d (ctr=0.49%, avg_position=10.5).

 5. content_5feee3994adb  (refresh_review)
    why: reason_code=stale_but_visible: not updated in 194 days, still pulling 7,812 impressions/90d (ctr=0.01%, avg_position=39.0).

 6. content_c2d929d83eaa  (refresh_review)
    why: reason_code=stale_but_visible: not updated in 193 days, sti

**What would make each one wrong** (one line each, cross-checked against that row's own numbers):

1. **Rank 1** (194 days stale, 61,678 impr., ctr 0.15%, pos 19.7) — wrong if the client already refreshed this page after the export date; my snapshot can't see updates that happened after it was pulled.
2. **Rank 2** (194 days, 59,472 impr., ctr 0.13%, pos 24.8) — wrong if the low ctr is really a SERP-feature issue (a featured snippet stealing clicks above it), not a content-quality problem a refresh would fix.
3. **Rank 3** (194 days, 25,715 impr., ctr 0.23%, pos 22.2) — wrong if `trend_direction=down` here reflects a seasonal keyword dip rather than genuine decay; a 90-day window can catch one bad season.
4. **Rank 4** (193 days, 13,299 impr., ctr 0.49%, pos 10.5) — this one already sits on page 1 (pos 10.5) with a decent ctr; wrong if it's actually fine and the `down` label is noise around a strong, stable page — lowest-confidence pick in the top 10.
5. **Rank 5** (194 days, 7,812 impr., ctr 0.01%, pos 39.0) — ctr near zero at position 39 could just mean position, not content, is the real problem; a refresh alone might not move it.
6. **Rank 6** (193 days, 7,558 impr., ctr 0.20%, pos 17.9) — wrong if this page's traffic is trending down repo-wide for the whole `client_7f2253d7e2` account (a client-level cause), not a page-level content problem this rule can fix.
7. **Rank 7** (194 days, 4,590 impr., ctr 0.00%, pos 31.0) — 0.00% ctr on real impressions is worth a manual look before spending refresh time; could be a tracking gap rather than a real content issue.
8. **Rank 8** (194 days, 4,556 impr., ctr 0.33%, pos 16.4) — wrong if `word_count` or `content_type` here means this is a thin/list-style page where more words won't help engagement.
9. **Rank 9** (194 days, 4,429 impr., ctr 0.38%, pos 25.3) — mid-pack pick; wrong if position 25.3 means it's actually stable at "page 3" and the `down` label reflects only the most recent 30 days.
10. **Rank 10** (193 days, 1,697 impr., ctr 0.12%, pos 15.8) — smallest-volume page in the top 10; wrong if the raw-impressions tiebreaker overweights it relative to pages with better ctr but slightly fewer impressions just outside the top 10.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# --- Weak-pick pattern check: client concentration ---
print("Client mix among ALL 17 flagged pages:")
print(df.loc[df['score'] > 0, 'client_id'].value_counts())
print("\nClient mix in just the top 10 (by score):")
print(top10["client_id"].value_counts())


Client mix among ALL 17 flagged pages:
client_id
client_7f2253d7e2    12
client_d029fa3a95     3
client_4ec9599fc2     1
client_9400f1b21c     1
Name: count, dtype: int64

Client mix in just the top 10 (by score):
client_id
client_7f2253d7e2    10
Name: count, dtype: int64


**Weak pick, named plainly:** the rule flags pages from 4 different clients (12 / 3 / 1 / 1), but because I rank the flagged group by raw `impressions_90d`, all 10 of my top-10 slots go to one client's biggest pages. A ranker that only ever surfaces one client's content isn't a useful portfolio-wide queue — it's an accident of one client happening to run the largest pages. A steadier version would rank *within* client first, or use a log/percentile of impressions instead of the raw count, so the queue doesn't get monopolized by whoever has the biggest traffic.

In [9]:
# --- Leakage check ---
# 1. Confirm the label-source columns never entered the score.
label_source_cols = {"trend_direction", "trend_pct", "is_declining_label"}
score_inputs = {"days_since_last_update", "impressions_90d"}
print("Score inputs:", score_inputs)
print("Overlap with label-source columns (should be empty):", score_inputs & label_source_cols)
assert len(score_inputs & label_source_cols) == 0, "Label-derived column leaked into the score!"

# 2. Confirm no FlyRank product/decision flags exist as columns in this starter slice at all --
# they were deliberately left out of the export (see 02_your_first_readable_model.ipynb, section 3).
suspicious = [c for c in df.columns if any(
    p in c.lower() for p in ["health_score", "recommended_action", "action_type", "flag"]
)]
print("\nColumns matching product-flag-like names (should be empty):", suspicious)

# 3. This is a single trailing-90-day snapshot, not a train/test time split -- there is no future
#    window to leak from within this file. days_since_last_update and impressions_90d are both
#    facts observable as of the export date, same moment the label itself is computed from.
print("\nNo label-derived or future-window inputs in the score. Confirmed.")


Score inputs: {'impressions_90d', 'days_since_last_update'}
Overlap with label-source columns (should be empty): set()

Columns matching product-flag-like names (should be empty): []

No label-derived or future-window inputs in the score. Confirmed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.